In [ ]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import sys
sys.path.insert(0, '../src')

import json
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from skimage import measure

import ipywidgets as widgets
from IPython.display import display, clear_output

from notebooks.utils.visualizer_utils import (
    hu_to_display, COLOR_AXIAL, COLOR_CORONAL, COLOR_SAGITTAL,
    load_decisions, save_decisions, overlay_multi_layer_mask,
    render_orthogonal_views, decision_badge, sync_sliders, sync_case_index,
    create_layer_checkboxes, create_overlay_control_panel, DEFAULT_LAYER_CONFIG,
    get_visible_labels, build_layer_status_suffix, create_viewer_layout,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = Path('../data/dataset/Dataset820/voi')
IMAGES_DIR     = OUTPUT_DIR / 'images'
MASKS_DIR      = OUTPUT_DIR / 'mask'
DELETED_ROOT   = OUTPUT_DIR / 'deleted'
DECISIONS_FILE = OUTPUT_DIR / 'decisions.json'

# ── Protocol set (used for buttons and discovery) ─────────────────────────────
KNOWN_PROTOCOLS = ['NC', 'ART', 'VEN']

# ── HU display range ──────────────────────────────────────────────────────────
HU_MIN = -150
HU_MAX =  250

# ── Overlay appearance ────────────────────────────────────────────────────────
# Centroid fallback target in legacy logic (tumor label)
MASK_LABEL = 2

# Multi-layer configuration (kidney=1, tumor=2, cyst=3)
LAYER_CONFIG = DEFAULT_LAYER_CONFIG.copy()
DEFAULT_VISIBLE = {1: True, 2: True, 3: False}  # Show kidney & tumor by default

# Phase filter for discovery
PHASE_FILTER = "ALL"  # Show all phases by default

# ── Sanity check ──────────────────────────────────────────────────────────────
assert IMAGES_DIR.exists(), f"images/ not found at {IMAGES_DIR}"
assert MASKS_DIR.exists(),  f"mask/ not found at {MASKS_DIR}"

print("✓ Imports and configuration ready")
print(f"  OUTPUT_DIR : {OUTPUT_DIR.resolve()}")
print(f"  IMAGES_DIR : {IMAGES_DIR.resolve()}")
print(f"  MASKS_DIR  : {MASKS_DIR.resolve()}")
print(f"  DECISIONS  : {DECISIONS_FILE}")
print(f"  HU range   : [{HU_MIN}, {HU_MAX}] → normalised [0, 1]")


In [ ]:
# ── Cell 2: Discover Cases & Load Existing Decisions ─────────────────────────

# ── Discover all image files ──────────────────────────────────────────────────
def discover_cases(phase_filter: str = 'ALL') -> list[dict]:
    """
    Walk IMAGES_DIR/<class>/<patient_id>/<protocol>/*.npy and build a flat
    list of case dicts, pairing each image with its corresponding mask.
    """
    cases = []
    for cls_dir in sorted(IMAGES_DIR.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls = cls_dir.name
        for patient_dir in sorted(cls_dir.iterdir()):
            if not patient_dir.is_dir():
                continue
            patient_id = patient_dir.name
            for proto_dir in sorted(patient_dir.iterdir()):
                if not proto_dir.is_dir():
                    continue
                protocol = proto_dir.name.upper()
                if phase_filter != 'ALL' and protocol != phase_filter.upper():
                    continue
                for img_path in sorted(proto_dir.glob('*.npy')):
                    filename  = img_path.name
                    mask_path = MASKS_DIR / cls / patient_id / proto_dir.name / filename
                    cases.append({
                        'class':      cls,
                        'patient_id': patient_id,
                        'protocol':   protocol,
                        'filename':   filename,
                        'image_path': img_path,
                        'mask_path':  mask_path,
                        # Unique key used for decisions.json
                        'key': f"{cls}/{patient_id}/{proto_dir.name}/{filename}",
                    })
    return cases

all_cases = discover_cases(PHASE_FILTER)

# ── Load existing decisions from disk ─────────────────────────────────────────
decisions = load_decisions(DECISIONS_FILE)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"✓ Discovered {len(all_cases)} cases  (filter: {PHASE_FILTER})")

proto_counts = defaultdict(int)
missing_masks = 0
for c in all_cases:
    proto_counts[c['protocol']] += 1
    if not c['mask_path'].exists():
        missing_masks += 1

for proto, n in sorted(proto_counts.items()):
    print(f"    {proto}: {n} cases")

if missing_masks:
    print(f"  ⚠ {missing_masks} cases have no matching mask file")
else:
    print(f"  ✓ All masks present")

print(f"\n✓ Loaded {len(decisions)} existing decisions from {DECISIONS_FILE.name}")
decided = defaultdict(int)
for e in decisions.values():
    decided[e['action']] += 1
for action, n in sorted(decided.items()):
    print(f"    {action}: {n}")


In [ ]:
# ── Cell 3: State Class ───────────────────────────────────────────────────────

class CaseReviewState:
    """
    Holds the full case list, current position, loaded arrays, and slice indices.

    Array convention (from numpy .npy files): shape = (X, Y, Z)
      Axial    — image[:, :, az]   viewed along Z
      Coronal  — image[:, cy, :]   viewed along Y
      Sagittal — image[sx, :, :]   viewed along X
    """

    def __init__(self, cases: list, decisions: dict):
        self.cases     = cases
        self.decisions = decisions          # shared ref — mutations persist
        self.n_cases   = len(cases)
        self.idx       = 0                  # current case index

        # Loaded arrays (filled by load_current)
        self.image: np.ndarray | None = None
        self.mask:  np.ndarray | None = None
        self.disp:  np.ndarray | None = None  # hu_to_display(image)

        # Current slice indices (updated when a new case is loaded)
        self.ax_idx:  int = 0   # axial   (Z axis)
        self.cor_idx: int = 0   # coronal (Y axis)
        self.sag_idx: int = 0   # sagittal (X axis)

    # ── Navigation ────────────────────────────────────────────────────────────

    def current_case(self) -> dict | None:
        if 0 <= self.idx < self.n_cases:
            return self.cases[self.idx]
        return None

    def go_next(self):
        if self.idx < self.n_cases - 1:
            self.idx += 1
            self.load_current()

    def go_prev(self):
        if self.idx > 0:
            self.idx -= 1
            self.load_current()

    # ── Loading ───────────────────────────────────────────────────────────────

    def load_current(self) -> bool:
        """Load image + mask for the current case; set slice indices to mask centroid."""
        case = self.current_case()
        if case is None:
            return False

        try:
            self.image = np.load(case['image_path'])   # shape (X, Y, Z)
            if case['mask_path'].exists():
                self.mask = np.load(case['mask_path'])
            else:
                # No mask: create an empty one with the same shape
                self.mask = np.zeros_like(self.image, dtype=np.uint8)

            self.disp = hu_to_display(self.image, HU_MIN, HU_MAX)
            self._set_center_slices()
            return True

        except Exception as e:
            print(f"Error loading {case['key']}: {e}")
            self.image = self.mask = self.disp = None
            return False

    def _set_center_slices(self):
        """Set ax/cor/sag indices to the centroid of mask == MASK_LABEL (or volume centre)."""
        if self.mask is None or self.image is None:
            if self.image is not None:
                X, Y, Z = self.image.shape
            else:
                X, Y, Z = (1, 1, 1)
            self.sag_idx = X // 2
            self.cor_idx = Y // 2
            self.ax_idx  = Z // 2
            return

        coords = np.argwhere(self.mask == MASK_LABEL)
        if coords.size > 0:
            centroid = coords.mean(axis=0).astype(int)
            self.sag_idx = int(np.clip(centroid[0], 0, self.image.shape[0] - 1))
            self.cor_idx = int(np.clip(centroid[1], 0, self.image.shape[1] - 1))
            self.ax_idx  = int(np.clip(centroid[2], 0, self.image.shape[2] - 1))
        else:
            X, Y, Z = self.image.shape
            self.sag_idx = X // 2
            self.cor_idx = Y // 2
            self.ax_idx  = Z // 2

    # ── Shapes / bounds ────────────────────────────────────────────────────────

    @property
    def shape(self):
        if self.image is not None:
            return self.image.shape
        return (1, 1, 1)

    def max_ax(self):  return self.shape[2] - 1
    def max_cor(self): return self.shape[1] - 1
    def max_sag(self): return self.shape[0] - 1

    # ── Decision helpers ──────────────────────────────────────────────────────

    def get_decision(self) -> str | None:
        case = self.current_case()
        if case and case['key'] in self.decisions:
            return self.decisions[case['key']]['action']
        return None

    def set_decision(self, action: str):
        """Record a decision for the current case and immediately save to disk."""
        case = self.current_case()
        if case is None:
            return
        entry = {
            'key':        case['key'],
            'action':     action,
            'class':      case['class'],
            'patient_id': case['patient_id'],
            'protocol':   case['protocol'],
            'filename':   case['filename'],
        }
        self.decisions[case['key']] = entry
        save_decisions(self.decisions, DECISIONS_FILE)

    def clear_decision(self):
        """Remove decision for the current case (used when toggling between actions)."""
        case = self.current_case()
        if case and case['key'] in self.decisions:
            del self.decisions[case['key']]
            save_decisions(self.decisions, DECISIONS_FILE)


# ── Instantiate ───────────────────────────────────────────────────────────────
state = CaseReviewState(all_cases, decisions)
ok    = state.load_current()

if ok:
    c = state.current_case()
    print(f"✓ State ready  ({state.n_cases} cases)")
    print(f"  First case : {c['key']}")
    print(f"  Array shape: {state.shape}  (X, Y, Z)")
    print(f"  Center slice → axial={state.ax_idx}, coronal={state.cor_idx}, sagittal={state.sag_idx}")
else:
    print("⚠ No cases loaded — check PHASE_FILTER and directory structure")


In [ ]:
# ── Cell 4: Render Function ───────────────────────────────────────────────────

def _overlay_mask_multi(ax_obj, mask_slice_2d: np.ndarray, visible_labels: dict):
    """Multi-layer overlay wrapper for voi masks."""
    overlay_multi_layer_mask(ax_obj, mask_slice_2d, visible_labels, LAYER_CONFIG)


def render(output_widget: widgets.Output):
    """Render axial | coronal | sagittal into the given Output widget."""
    active_checkboxes = globals().get('layer_checkboxes', None)
    if active_checkboxes is None:
        visible_labels = DEFAULT_VISIBLE.copy()
    else:
        visible_labels = get_visible_labels(active_checkboxes)

    suffix = build_layer_status_suffix(
        visible_labels=visible_labels,
        layer_config=LAYER_CONFIG,
        has_overlay=(state.mask is not None),
    )

    render_orthogonal_views(
        state,
        output_widget,
        overlay_func=_overlay_mask_multi,
        case_title_suffix=suffix,
        visible_labels=visible_labels,
    )


# ── Quick test render ─────────────────────────────────────────────────────────
_test_out = widgets.Output()
display(_test_out)
render(_test_out)
print("✓ Render function ready")


In [ ]:
# ── Cell 5: Widgets & Callbacks ──────────────────────────────────────────────

_updating = [False]   # guard flag in list so it can be modified by helper functions

# ── Output widget (holds the matplotlib figure) ───────────────────────────────
out = widgets.Output()

# ── Status label ──────────────────────────────────────────────────────────────
status_html = widgets.HTML()

# ── Layer control checkboxes ──────────────────────────────────────────────────
layer_checkboxes = create_layer_checkboxes(LAYER_CONFIG, DEFAULT_VISIBLE)
overlay_panel = create_overlay_control_panel(layer_checkboxes, LAYER_CONFIG)

# ── Case index text box ───────────────────────────────────────────────────────
txt_case_idx = widgets.BoundedIntText(
    value=1,
    min=1,
    max=len(all_cases),
    step=1,
    description='Case #:',
    layout=widgets.Layout(width='180px')
)

# ── Slice sliders ─────────────────────────────────────────────────────────────
_slider_layout = widgets.Layout(width='50%')

sl_ax  = widgets.IntSlider(description='Axial (Z)',   min=0, max=1, step=1,
                            continuous_update=True, layout=_slider_layout)
sl_cor = widgets.IntSlider(description='Coronal (Y)', min=0, max=1, step=1,
                            continuous_update=True, layout=_slider_layout)
sl_sag = widgets.IntSlider(description='Sagittal (X)', min=0, max=1, step=1,
                             continuous_update=True, layout=_slider_layout)

# ── Buttons ───────────────────────────────────────────────────────────────────
btn_layout   = widgets.Layout(width='90px',  height='44px')
btn_layout_w = widgets.Layout(width='105px', height='44px')

btn_prev   = widgets.Button(description='← Prev',   button_style='info',    layout=btn_layout)
btn_next   = widgets.Button(description='Next →',   button_style='info',    layout=btn_layout)
btn_nc     = widgets.Button(description='NC',        button_style='success', layout=btn_layout)
btn_art    = widgets.Button(description='ART',       button_style='success', layout=btn_layout)
btn_ven    = widgets.Button(description='VEN',       button_style='success', layout=btn_layout)
btn_skip   = widgets.Button(description='Skip',      button_style='warning', layout=btn_layout)
btn_delete = widgets.Button(description='✗ Delete',  button_style='danger',  layout=btn_layout_w)

# ── Helpers ───────────────────────────────────────────────────────────────────

def _sync_sliders():
    """Push current state slice indices into the sliders (without triggering a render)."""
    sync_sliders(state, sl_ax, sl_cor, sl_sag, _updating)

def _sync_case_index():
    """Update the case index text box without triggering a callback."""
    sync_case_index(state, txt_case_idx, _updating)

def _update_status():
    case     = state.current_case()
    decision = state.get_decision()
    if case is None:
        status_html.value = "<b style='color:green;font-size:14px;'>✓ All cases reviewed!</b>"
        return
    total_decided = len(state.decisions)
    status_html.value = f"""
    <div style='background:#1e1e1e;color:#ddd;padding:8px 12px;border-radius:6px;
                font-family:monospace;font-size:12px;line-height:1.7;'>
      <b style='font-size:13px;color:#fff;'>
        Case {state.idx + 1} / {state.n_cases}</b>
      &nbsp;·&nbsp; {decision_badge(decision)}<br/>
      <span style='color:#aaa;'>key:</span> {case['key']}<br/>
      <span style='color:#aaa;'>protocol:</span> {case['protocol']}
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>class:</span> {case['class']}
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>patient:</span> {case['patient_id']}<br/>
      <span style='color:#aaa;'>decisions saved:</span> {total_decided}
    </div>"""

def _full_refresh():
    """Sync sliders, update status bar, and re-render the figure."""
    _sync_sliders()
    _sync_case_index()
    _update_status()
    render(out)

# ── Slider callbacks ──────────────────────────────────────────────────────────

def _on_ax_change(change):
    if _updating[0]:
        return
    state.ax_idx = change['new']
    render(out)

def _on_cor_change(change):
    if _updating[0]:
        return
    state.cor_idx = change['new']
    render(out)

def _on_sag_change(change):
    if _updating[0]:
        return
    state.sag_idx = change['new']
    render(out)

sl_ax.observe(_on_ax_change,  names='value')
sl_cor.observe(_on_cor_change, names='value')
sl_sag.observe(_on_sag_change, names='value')

# ── Layer checkbox callbacks ──────────────────────────────────────────────────

def _on_layer_toggle(change):
    """Re-render when any layer checkbox is toggled."""
    if _updating[0]: return
    render(out)

for checkbox in layer_checkboxes.values():
    checkbox.observe(_on_layer_toggle, names='value')

# ── Case index text box callback ──────────────────────────────────────────────

def _on_case_idx_change(change):
    """Navigate to the specified case index (1-indexed)."""
    if _updating[0]: return
    target_idx = change['new'] - 1  # Convert to 0-indexed
    if 0 <= target_idx < state.n_cases:
        state.idx = target_idx
        state.load_current()
        _full_refresh()
    else:
        # Invalid index, revert to current
        _sync_case_index()

txt_case_idx.observe(_on_case_idx_change, names='value')

# ── Button callbacks ──────────────────────────────────────────────────────────

def _classify(protocol: str):
    """Record a classify decision (NC/ART/VEN), advance, refresh."""
    state.set_decision(protocol)   # overwrites any prior delete entry
    state.go_next()
    _full_refresh()

def _on_nc(_):     _classify('NC')
def _on_art(_):    _classify('ART')
def _on_ven(_):    _classify('VEN')

def _on_delete(_):
    """Record a delete decision, advance, refresh."""
    state.set_decision('delete')   # overwrites any prior classify entry
    state.go_next()
    _full_refresh()

def _on_skip(_):
    """Advance without recording any decision."""
    state.go_next()
    _full_refresh()

def _on_next(_):
    """Navigate forward (does NOT record a decision)."""
    state.go_next()
    _full_refresh()

def _on_prev(_):
    """Navigate backward (does NOT record a decision)."""
    state.go_prev()
    _full_refresh()

btn_nc.on_click(_on_nc)
btn_art.on_click(_on_art)
btn_ven.on_click(_on_ven)
btn_delete.on_click(_on_delete)
btn_skip.on_click(_on_skip)
btn_next.on_click(_on_next)
btn_prev.on_click(_on_prev)

# ── Initial sync ──────────────────────────────────────────────────────────────
_sync_sliders()
_sync_case_index()
_update_status()

print("✓ Widgets and callbacks ready — run cell 6 to display the UI")


In [ ]:
# ── Cell 6: Display UI ────────────────────────────────────────────────────────

# Navigation row with case index text box
_nav_row = widgets.HBox(
    [btn_prev, txt_case_idx, btn_next],
    layout=widgets.Layout(
        justify_content='center',
        gap='8px',
        padding='6px 0',
    )
)

_btn_row = widgets.HBox(
    [btn_nc, btn_art, btn_ven, btn_skip, btn_delete],
    layout=widgets.Layout(
        justify_content='center',
        gap='6px',
        padding='6px 0',
    )
)

# Sliders at 50% width, centered horizontally
_sliders = widgets.VBox(
    [sl_ax, sl_cor, sl_sag],
    layout=widgets.Layout(
        align_items='center',
        width='100%',
        padding='4px 0',
    )
)

ui = create_viewer_layout(
    status_html=status_html,
    out=out,
    overlay_panel=overlay_panel,
    sliders=_sliders,
    nav_row=_nav_row,
    btn_row=_btn_row,
)

# Initial render
_full_refresh()
display(ui)


## Apply Decisions

Run cells 7, 8, and 9 **after** you have finished reviewing cases in the viewer above.  
Each cell loads `decisions.json` independently — no kernel state from the viewer is required.

| Cell | Action |
|---|---|
| 7 | Move files to their new protocol folder (NC / ART / VEN reclassifications) |
| 8 | Move files marked `delete` to `deleted/` subfolder (non-destructive) |
| 9 | Remove empty directories left behind |


In [ ]:
# ── Cell 7: Apply Reclassifications ──────────────────────────────────────────
#
# Reads decisions.json and moves image + mask files for every entry whose
# action is NC, ART, or VEN from the original protocol folder to the new one.
#
# Source : <IMAGES_DIR|MASKS_DIR>/<class>/<patient_id>/<old_protocol>/<filename>
# Target : <IMAGES_DIR|MASKS_DIR>/<class>/<patient_id>/<new_protocol>/<filename>
#
# The operation is idempotent: if the source no longer exists (already moved),
# the entry is skipped with a warning rather than raising an error.

import json, shutil
from pathlib import Path
from collections import defaultdict

_OUTPUT_DIR     = Path('../data/dataset/Dataset720/voi')
_IMAGES_DIR     = _OUTPUT_DIR / 'images'
_MASKS_DIR      = _OUTPUT_DIR / 'mask'
_DECISIONS_FILE = _OUTPUT_DIR / 'decisions.json'

# ── Load decisions ────────────────────────────────────────────────────────────
assert _DECISIONS_FILE.exists(), f"decisions.json not found at {_DECISIONS_FILE}"

with open(_DECISIONS_FILE, 'r') as _f:
    _all_decisions = json.load(_f)

_reclassify = [e for e in _all_decisions
               if e.get('action', '').upper() in ('NC', 'ART', 'VEN')]

print(f"decisions.json  : {len(_all_decisions)} total entries")
print(f"to reclassify   : {len(_reclassify)}\n")

# ── Move files ────────────────────────────────────────────────────────────────
moved_files     = []
skipped_missing = []

for entry in _reclassify:
    cls        = entry['class']
    patient_id = entry['patient_id']
    old_proto  = entry['protocol']          # original protocol folder name
    new_proto  = entry['action'].upper()    # target protocol folder name
    filename   = entry['filename']

    if old_proto == new_proto:
        continue  # already in the right place

    for base_dir in [_IMAGES_DIR, _MASKS_DIR]:
        src = base_dir / cls / patient_id / old_proto / filename
        dst_dir = base_dir / cls / patient_id / new_proto
        dst = dst_dir / filename

        if dst.exists():
            # Already moved in a previous run — update key only
            continue
        if not src.exists():
            skipped_missing.append(str(src.relative_to(_OUTPUT_DIR)))
            continue

        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        moved_files.append({
            'from': str(src.relative_to(_OUTPUT_DIR)),
            'to':   str(dst.relative_to(_OUTPUT_DIR)),
        })

# ── Update keys in decisions.json to reflect new protocol ────────────────────
updated_count = 0
for entry in _all_decisions:
    if entry.get('action', '').upper() not in ('NC', 'ART', 'VEN'):
        continue
    old_proto = entry['protocol']
    new_proto = entry['action'].upper()
    if old_proto == new_proto:
        continue
    # Rebuild key: <class>/<patient_id>/<new_proto>/<filename>
    entry['key']      = f"{entry['class']}/{entry['patient_id']}/{new_proto}/{entry['filename']}"
    entry['protocol'] = new_proto
    updated_count += 1

with open(_DECISIONS_FILE, 'w') as _f:
    json.dump(_all_decisions, _f, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"✓ Files moved        : {len(moved_files)}")
if skipped_missing:
    print(f"⚠ Skipped (missing) : {len(skipped_missing)}")
    for p in skipped_missing:
        print(f"    {p}")
print(f"✓ decisions.json keys updated : {updated_count}")

# Break down by new protocol
by_proto = defaultdict(int)
for e in moved_files:
    by_proto[Path(e['to']).parts[2]] += 1
for proto, n in sorted(by_proto.items()):
    print(f"    → {proto}: {n} files")


In [ ]:
# ── Cell 8: Execute Deletions (non-destructive) ───────────────────────────────
#
# Files marked with action == 'delete' are MOVED (never unlink'd) to:
#   OUTPUT_DIR/deleted/images/<class>/<patient_id>/<protocol>/<filename>
#   OUTPUT_DIR/deleted/mask/<class>/<patient_id>/<protocol>/<filename>
#
# You can restore any file by moving it back manually.
# A deletion log is saved to OUTPUT_DIR/deletion_log.json.

import json, shutil
from pathlib import Path

_OUTPUT_DIR     = Path('../data/dataset/Dataset720/voi')
_IMAGES_DIR     = _OUTPUT_DIR / 'images'
_MASKS_DIR      = _OUTPUT_DIR / 'mask'
_DELETED_ROOT   = _OUTPUT_DIR / 'deleted'
_DECISIONS_FILE = _OUTPUT_DIR / 'decisions.json'

# ── Load decisions ────────────────────────────────────────────────────────────
assert _DECISIONS_FILE.exists(), f"decisions.json not found at {_DECISIONS_FILE}"

with open(_DECISIONS_FILE, 'r') as _f:
    _all_decisions = json.load(_f)

_to_delete = [e for e in _all_decisions if e.get('action') == 'delete']

print(f"decisions.json   : {len(_all_decisions)} total entries")
print(f"marked delete    : {len(_to_delete)}\n")

if not _to_delete:
    print("Nothing to do.")
else:
    moved_to_deleted = []
    skipped_missing  = []

    for entry in _to_delete:
        cls        = entry['class']
        patient_id = entry['patient_id']
        protocol   = entry['protocol']
        filename   = entry['filename']

        for base_dir, label in [(_IMAGES_DIR, 'images'), (_MASKS_DIR, 'mask')]:
            src     = base_dir / cls / patient_id / protocol / filename
            dst_dir = _DELETED_ROOT / label / cls / patient_id / protocol
            dst     = dst_dir / filename

            if dst.exists():
                continue  # already moved in a previous run
            if not src.exists():
                skipped_missing.append(str(src.relative_to(_OUTPUT_DIR)))
                continue

            dst_dir.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            moved_to_deleted.append({
                'from': str(src.relative_to(_OUTPUT_DIR)),
                'to':   str(dst.relative_to(_OUTPUT_DIR)),
            })
            print(f"  Moved: {src.relative_to(_OUTPUT_DIR)}")

    # ── Save deletion log ─────────────────────────────────────────────────────
    _log_path = _OUTPUT_DIR / 'deletion_log.json'
    _existing_log = []
    if _log_path.exists():
        with open(_log_path, 'r') as _f:
            _existing_log = json.load(_f).get('moved_files', [])

    with open(_log_path, 'w') as _f:
        json.dump({
            'total_entries':  len(_to_delete),
            'moved_files':    _existing_log + moved_to_deleted,
        }, _f, indent=2)

    print(f"\n✓ Moved to deleted/ : {len(moved_to_deleted)} files")
    if skipped_missing:
        print(f"⚠ Skipped (missing): {len(skipped_missing)}")
        for p in skipped_missing:
            print(f"    {p}")
    print(f"✓ Deletion log      : {_log_path.relative_to(_OUTPUT_DIR)}")


In [ ]:
# ── Cell 9: Clean Up Empty Folders ───────────────────────────────────────────
#
# After reclassifications and deletions, some protocol subfolders may be empty.
# This cell removes them bottom-up under images/ and mask/.
# The deleted/ folder and OUTPUT_DIR root are never touched.

from pathlib import Path

_OUTPUT_DIR = Path('../data/dataset/Dataset720/voi')

def _remove_empty_folders(root: Path) -> list[str]:
    """Recursively remove empty subdirectories under root (root itself is kept)."""
    removed = []
    # Sort deepest paths first so children are removed before parents
    for folder in sorted(root.rglob('*'), key=lambda p: len(p.parts), reverse=True):
        if not folder.is_dir() or folder == root:
            continue
        try:
            if not any(folder.iterdir()):
                folder.rmdir()
                removed.append(str(folder.relative_to(_OUTPUT_DIR)))
                print(f"  Removed: {folder.relative_to(_OUTPUT_DIR)}")
        except Exception as e:
            print(f"  Could not remove {folder}: {e}")
    return removed

print("Scanning images/ and mask/ for empty folders …\n")

removed = []
for _root in [_OUTPUT_DIR / 'images', _OUTPUT_DIR / 'mask']:
    if _root.exists():
        removed.extend(_remove_empty_folders(_root))

print(f"\n✓ Removed {len(removed)} empty folder(s)")
